In [7]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)


query = "SELECT * FROM kerberos_events"

df = pd.read_sql(query, engine)

df

,id,event_time,username,source_ip,host,event_type,status,service_name,ticket_encryption,failure_reason
0,1,2026-04-09 08:10:00,david,10.0.1.15,dc01,TGT_REQUEST,SUCCESS,krbtgt,AES256,NaN
1,2,2026-04-09 08:12:00,david,10.0.1.15,dc01,TGT_REQUEST,SUCCESS,krbtgt,AES256,NaN
2,3,2026-04-09 08:15:00,sara,10.0.1.20,dc01,TGT_REQUEST,SUCCESS,krbtgt,AES256,NaN
3,4,2026-04-09 08:20:00,itay,10.0.1.25,dc01,TGT_REQUEST,FAILURE,krbtgt,AES256,PREAUTH_FAILED
4,5,2026-04-09 08:21:00,itay,10.0.1.25,dc01,TGT_REQUEST,FAILURE,krbtgt,AES256,PREAUTH_FAILED
5,6,2026-04-09 08:22:00,itay,10.0.1.25,dc01,TGT_REQUEST,FAILURE,krbtgt,AES256,PREAUTH_FAILED
6,7,2026-04-09 08:23:00,itay,10.0.1.25,dc01,TGT_REQUEST,SUCCESS,krbtgt,AES256,NaN
7,8,2026-04-09 02:14:00,admin1,185.220.101.5,dc01,TGT_REQUEST,SUCCESS,krbtgt,RC4,NaN
8,9,2026-04-09 02:16:00,admin1,185.220.101.5,dc01,TGT_REQUEST,SUCCESS,krbtgt,RC4,NaN
9,10,2026-04-09 02:18:00,admin1,185.220.101.5,dc01,TGT_REQUEST,SUCCESS,krbtgt,RC4,NaN


In [4]:
alerts = []

time_window = pd.Timedelta(minutes=15)

for username, group in df.groupby("username"):
    failures = 0
    failure_time = None

    group = group.sort_values("event_time").reset_index(drop=True)

    for i in range(len(group)):
        event = group.iloc[i]

        if event["status"] == "FAILURE":
            if failures == 0:
                failure_time = event["event_time"]
            failures += 1

        elif event["status"] == "SUCCESS":
            if failures >= 3 and failure_time is not None:
                if event["event_time"] <= failure_time + time_window:
                    alerts.append({
                        "alert_type": "Kerberos brute-force success pattern",
                        "username": username,
                        "failures_before_success": failures,
                        "time": str(event["event_time"])
                    })                  
                    break

alerts

[{'alert_type': 'Kerberos brute-force success pattern',
  'username': 'itay',
  'failures_before_success': 3,
  'time': '2026-04-09 08:23:00'}]

In [5]:

time_window = pd.Timedelta(minutes=15)

for username, group in df.groupby("username"):

    group = group.sort_values("event_time").reset_index(drop=True)

    for i in range(len(group)):

        start_time = group.iloc[i]["event_time"]

        window = group[
            (group["event_time"] >= start_time) &
            (group["event_time"] <= start_time + time_window)
        ]
        unique_ips = window["source_ip"].nunique()

        if unique_ips > 2:

            alerts.append({
                "alert_type": "Kerberos multi-source TGT activity",
                "username": username,
                "unique_source_ip_count": unique_ips,
                "window_start": str(start_time)
            })

            break
alerts

[{'alert_type': 'Kerberos brute-force success pattern',
  'username': 'itay',
  'failures_before_success': 3,
  'time': '2026-04-09 08:23:00'},
 {'alert_type': 'Kerberos multi-source TGT activity',
  'username': 'michael',
  'unique_source_ip_count': 4,
  'window_start': '2026-04-09 09:00:00'}]

In [6]:
import json

for _, event in df.iterrows():
    if event["ticket_encryption"] == "RC4":
        alerts.append({
            "alert_type": "Kerberos RC4 encryption usage",
            "username": event["username"],
            "source_ip": event["source_ip"],
            "event_type": event["event_type"],
            "time": str(event["event_time"])
        })
        
output_data = {
    "total_alerts": len(alerts),
    "alerts": alerts
}

with open("outputs/alerts_from_kerberos_detector.json", "w") as f:
    json.dump(output_data, f, indent=2)
alerts

[{'alert_type': 'Kerberos brute-force success pattern',
  'username': 'itay',
  'failures_before_success': 3,
  'time': '2026-04-09 08:23:00'},
 {'alert_type': 'Kerberos multi-source TGT activity',
  'username': 'michael',
  'unique_source_ip_count': 4,
  'window_start': '2026-04-09 09:00:00'},
 {'alert_type': 'Kerberos RC4 encryption usage',
  'username': 'admin1',
  'source_ip': '185.220.101.5',
  'event_type': 'TGT_REQUEST',
  'time': '2026-04-09 02:14:00'},
 {'alert_type': 'Kerberos RC4 encryption usage',
  'username': 'admin1',
  'source_ip': '185.220.101.5',
  'event_type': 'TGT_REQUEST',
  'time': '2026-04-09 02:16:00'},
 {'alert_type': 'Kerberos RC4 encryption usage',
  'username': 'admin1',
  'source_ip': '185.220.101.5',
  'event_type': 'TGT_REQUEST',
  'time': '2026-04-09 02:18:00'}]